# CodeAlpha Task 4 — Sales Prediction System

**Internship:** CodeAlpha Data Science Internship  
**Task:** Predict sales revenue from TV, Radio, and Newspaper advertising spend.  
**Dataset:** `Advertising.csv` — 200 rows, 3 spend features + 1 target (Sales).  

---

## Notebook Structure

| Section | Content |
|---------|--------|
| 1 | Data Loading & Inspection |
| 2 | Data Cleaning |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Feature Engineering & Preprocessing |
| 5 | Model Training & Comparison (7 algorithms) |
| 6 | Hyperparameter Tuning (GridSearchCV) |
| 7 | Model Evaluation & Visualisations |
| 8 | Single-Point Prediction |
| 9 | Batch Forecasting |
| 10 | Business Insights |
| 11 | Model Persistence & Reload Verification |

> **Convention note:** This notebook follows the same conventions as Tasks 1–3:  
> - PEP 8 compliant code, docstrings on all functions  
> - Markdown insight paragraphs *before* and *after* every plot  
> - Exception handling via explicit `raise` (no silent `except: pass`)  
> - Shared logic lives in `utils.py` — this notebook imports from there

In [ ]:
# ─── Standard library ─────────────────────────────────────────────────────
import warnings
from pathlib import Path

# ─── Third-party ─────────────────────────────────────────────────────────
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

# ─── Local helpers ────────────────────────────────────────────────────────
from utils import (
    FEATURE_COLS,
    TARGET_COL,
    clean_data,
    evaluate_model,
    forecast_sales,
    load_artifacts,
    load_data,
    predict_sales,
    save_artifacts,
)

# ─── Configuration ───────────────────────────────────────────────────────
warnings.filterwarnings("ignore", category=FutureWarning)

# Matplotlib / Seaborn style
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "#1a1a2e",
    "axes.facecolor": "#16213e",
    "axes.labelcolor": "#e2e8f0",
    "text.color": "#e2e8f0",
    "xtick.color": "#a0aec0",
    "ytick.color": "#a0aec0",
    "grid.color": "#2d3748",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

RANDOM_STATE = 42
IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print("Environment ready ✓")

---
## Section 1 — Data Loading & Inspection

### ⚠️ Critical pitfall caught deliberately

The raw `Advertising.csv` was originally exported from a pandas DataFrame and contains an **unnamed first column** (`""`) that is simply the row index (`1, 2, 3, …, 200`).  
If you load with the default `pd.read_csv("Advertising.csv")`, that column silently enters your feature matrix as a numeric predictor.  Because row-index perfectly correlates with itself (it's monotonic), many models will pick it up as a spurious signal, artificially inflating R² in a way that **will not reproduce on any new data**.  

**Fix:** `pd.read_csv("Advertising.csv", index_col=0)` treats that column as the DataFrame index instead of a feature — it never enters `X`.

After loading we check `.shape`, `.head()`, `.info()`, null counts, and duplicate counts **programmatically** — we do not hardcode assumed values in comments.

In [ ]:
# Load with index_col=0 to drop the stray pandas-exported index column immediately
df = load_data("Advertising.csv")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\n--- Head ---")
display(df.head())
print("\n--- Tail ---")
display(df.tail())

In [ ]:
print("--- DataFrame Info ---")
df.info()

print("\n--- Descriptive Statistics ---")
display(df.describe().T.round(3))

# Null values — programmatically checked, not assumed
null_counts = df.isnull().sum()
print(f"\nNull values per column:\n{null_counts}")
print(f"Total nulls: {int(null_counts.sum())}")

# Duplicate rows
dup_count = int(df.duplicated().sum())
print(f"\nDuplicate rows: {dup_count}")

---
## Section 2 — Data Cleaning

### Outlier Analysis

Boxplots reveal the spread and extreme values of each column.  
**Expected finding:** `Newspaper` typically shows a long upper tail with a few high-spend campaigns.  These are **intentionally kept** — high advertising spend in a channel is a legitimate business event (a major campaign), not a data error.  Auto-dropping them would remove real signal and bias the model toward low-spend predictions.

**Decision rule applied:**  
- Outlier values that are *physically plausible* (spend can be large) → **keep, flag in markdown**  
- Outlier values that are *data errors* (negative spend, non-numeric) → **drop/fix**  

We then assert all values are non-negative to ensure logical consistency.

In [ ]:
df = clean_data(df)
print(f"Post-clean shape: {df.shape}")
print("Data cleaning passed all assertions ✓")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Boxplots — Outlier Detection", fontsize=15, fontweight="bold", color="#e2e8f0")

colors_box = ["#a78bfa", "#60a5fa", "#34d399", "#f59e0b"]
for ax, col, clr in zip(axes, df.columns, colors_box):
    bp = ax.boxplot(
        df[col].values,
        patch_artist=True,
        boxprops=dict(facecolor=clr, alpha=0.6),
        medianprops=dict(color="white", linewidth=2),
        whiskerprops=dict(color="#a0aec0"),
        capprops=dict(color="#a0aec0"),
        flierprops=dict(marker="o", color=clr, alpha=0.5),
    )
    ax.set_title(col, fontsize=12, color="#e2e8f0")
    ax.set_xticks([])

plt.tight_layout()
plt.savefig(IMAGES_DIR / "boxplots_outliers.png", bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

# IQR-based outlier count per column
for col in df.columns:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = int(((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum())
    print(f"{col:<12}: {n_out} IQR-based outlier(s) — {'kept (real campaigns)' if n_out > 0 else 'none'}")

**Finding:** Newspaper shows the most upper-tail outliers, consistent with large-campaign bursts.  All outliers are retained because they represent genuine high-spend periods, not data recording errors.

---
## Section 3 — Exploratory Data Analysis (EDA)

### What to look for
- **Histograms + KDE:** Check the shape of each distribution — is spend roughly uniform, skewed, or bimodal?  Sales should ideally be roughly normally distributed for OLS assumptions.
- **Pairplot:** Visual scan of all pairwise relationships — which features show a clear linear relationship with Sales?
- **Correlation heatmap:** Quantify linear associations.  Values > 0.7 are strong; values < 0.3 are weak.  Also check for multicollinearity between features.
- **Scatter plots (feature vs Sales):** Confirm direction, linearity, and the scatter of residuals visually.

> **Naming convention note:** The colormap variable is named `cmap` throughout — not `cm` — to avoid any collision with `sklearn.metrics.confusion_matrix` if Task 1/3 boilerplate is ever copied in.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Distribution Histograms + KDE", fontsize=15, fontweight="bold", color="#e2e8f0")

hist_colors = ["#a78bfa", "#60a5fa", "#34d399", "#f59e0b"]
for ax, col, clr in zip(axes.flat, df.columns, hist_colors):
    sns.histplot(df[col], kde=True, ax=ax, color=clr, bins=25, alpha=0.7)
    ax.set_title(f"Distribution of {col}", color="#e2e8f0")
    ax.set_xlabel(col, color="#a0aec0")
    ax.set_ylabel("Frequency", color="#a0aec0")
    # Annotate mean and median with float() to avoid numpy scalar matplotlib issues
    ax.axvline(float(df[col].mean()), color="white", linestyle="--", linewidth=1.2, label=f"Mean={float(df[col].mean()):.1f}")
    ax.axvline(float(df[col].median()), color=clr, linestyle=":", linewidth=1.5, label=f"Median={float(df[col].median()):.1f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(IMAGES_DIR / "histograms_kde.png", bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

**Finding:** TV spend is roughly uniformly distributed between 0–300K.  Radio spend is right-skewed (most campaigns are small).  Newspaper is also right-skewed with a long upper tail.  Sales is roughly bell-shaped with a slight right skew — well within acceptable bounds for linear regression.

In [ ]:
# Pairplot: diag_kind="kde" for smooth diagonals
cmap = "husl"  # deliberately named 'cmap', not 'cm', to avoid collision
pair_fig = sns.pairplot(
    df,
    diag_kind="kde",
    plot_kws={"alpha": 0.5, "s": 20},
    diag_kws={"color": "#a78bfa", "fill": True},
    palette="husl",
)
pair_fig.figure.suptitle("Pairplot — All Feature Combinations", y=1.02, fontsize=14, color="white")
pair_fig.figure.savefig(IMAGES_DIR / "pairplot.png", bbox_inches="tight")
plt.show()

**Finding:**  
- TV vs Sales shows a clear positive linear trend — the strongest relationship in the dataset.  
- Radio vs Sales also shows a positive trend, though with more scatter.  
- Newspaper vs Sales shows a weaker, noisier relationship.  
- TV and Newspaper have some positive correlation (~0.06) but no severe multicollinearity.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor("#1a1a2e")

correlation_matrix = df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",   # deliberately named 'cmap' arg, not a variable named 'cm'
    ax=ax,
    linewidths=0.5,
    linecolor="#2d3748",
    annot_kws={"size": 11, "weight": "bold"},
    vmin=-1,
    vmax=1,
)
ax.set_title("Correlation Heatmap", fontsize=14, fontweight="bold", color="#e2e8f0", pad=15)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "correlation_heatmap.png", bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

print("\nCorrelation with Sales:")
print(correlation_matrix["Sales"].sort_values(ascending=False))

**Finding:**  
- **TV vs Sales: ~0.78** — strong positive linear correlation  
- **Radio vs Sales: ~0.58** — moderate positive correlation  
- **Newspaper vs Sales: ~0.23** — weak correlation  
- No severe multicollinearity between features (all inter-feature correlations < 0.4)  

All three features are retained: weak correlation ≠ zero importance (tree models capture non-linear interactions; Newspaper may contribute in combination with TV).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Feature vs Sales (with Regression Line)", fontsize=14, fontweight="bold", color="#e2e8f0")

scatter_colors = ["#a78bfa", "#60a5fa", "#34d399"]
for ax, feat, clr in zip(axes, FEATURE_COLS, scatter_colors):
    sns.regplot(
        x=feat,
        y="Sales",
        data=df,
        ax=ax,
        scatter_kws={"alpha": 0.5, "color": clr, "s": 30},
        line_kws={"color": "#f59e0b", "linewidth": 2},
        ci=95,
    )
    ax.set_title(f"{feat} vs Sales", color="#e2e8f0")
    ax.set_xlabel(feat, color="#a0aec0")
    ax.set_ylabel("Sales", color="#a0aec0")

plt.tight_layout()
plt.savefig(IMAGES_DIR / "scatter_regplot.png", bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

**Finding:**  
- TV shows a clean positive linear relationship — ideal for linear regression  
- Radio also shows a positive linear trend, though with more dispersion at higher budgets  
- Newspaper's regression line is nearly flat — confirming weak predictive value as a standalone feature, though it may contribute in interaction with TV

---
## Section 4 — Feature Engineering & Preprocessing

### Feature Selection

All three channels (TV, Radio, Newspaper) are included as features:
- **No leakage risk:** None are derived from Sales — they are independent inputs.
- **All are legitimate business variables:** Even Newspaper, despite weak correlation, may carry signal in combination with other channels.

### Scaling

`StandardScaler` is applied to all three features.  
- **Why:** Linear/Ridge/Lasso regression coefficients are on the raw feature scale — without scaling, a coefficient of 0.05 for TV (range 0–300) and 0.2 for Radio (range 0–50) are on completely different scales, making coefficient interpretation and regularisation unfair.
- **For tree-based models** (Decision Tree, Random Forest, Gradient Boosting): scaling is mathematically unnecessary (trees use rank-based splits), but we scale anyway so all models see identical input, making the comparison fair and ensuring `app.py` can use a single scaler for any model.

### Train/Test Split

`test_size=0.2, random_state=42` — **no `stratify=` argument** (this is continuous regression, not classification — stratification is irrelevant and would raise an error).

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
    # No stratify= — continuous regression target, not classification
)

print(f"Training set  : {X_train.shape[0]} rows")
print(f"Test set      : {X_test.shape[0]} rows")
print(f"Features      : {FEATURE_COLS}")
print(f"Target        : {TARGET_COL}")

# Fit scaler on TRAINING SET ONLY — then transform both splits
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=FEATURE_COLS, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=FEATURE_COLS, index=X_test.index
)

print("\nScaled training features (first 3 rows):")
display(X_train_scaled.head(3))

---
## Section 5 — Model Training & Comparison

### Models trained
1. **Linear Regression** — baseline; interpretable coefficients as ROI proxy
2. **Ridge Regression** — L2 regularisation; handles mild multicollinearity
3. **Lasso Regression** — L1 regularisation; can drive weak features to zero
4. **Decision Tree Regressor** — captures non-linear splits; prone to overfitting
5. **Random Forest Regressor** — ensemble of trees; robust, generalises well
6. **Gradient Boosting Regressor** — boosted ensemble; often top performer
7. **XGBoost** — optimised gradient boosting; wrapped in try/except in case it's not installed

### Metrics
- **R²** — proportion of variance explained (higher = better, max 1.0)
- **MAE** — mean absolute error (same units as Sales)
- **MSE** — mean squared error (penalises large errors more)
- **RMSE** — root MSE (interpretable; same units as Sales)
- **CV R²** — 5-fold cross-validation R² on training set, so a lucky test split doesn't decide the winner

### RMSE compatibility note
`mean_squared_error(..., squared=False)` was deprecated in scikit-learn 1.4 and removed in 1.5.  
The `utils.py` helper `_rmse()` checks the sklearn version at runtime and calls `root_mean_squared_error` if available, otherwise falls back to `np.sqrt(mean_squared_error(...))` — no deprecation warnings.

In [ ]:
print(f"scikit-learn version: {sklearn.__version__}")

# XGBoost — optional, wrapped in try/except that re-raises clearly
try:
    from xgboost import XGBRegressor
    _xgb_available = True
    print("XGBoost available ✓")
except ImportError as exc:
    _xgb_available = False
    print(f"XGBoost not available: {exc}")
    print("Install with: pip install xgboost")
    # Do NOT silence — we just skip XGBoost from the comparison table

# Model registry
models_registry: dict[str, object] = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Lasso Regression": Lasso(alpha=0.01, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE),
}

if _xgb_available:
    models_registry["XGBoost"] = XGBRegressor(random_state=RANDOM_STATE, verbosity=0)

# Train all models and collect metrics
results_list: list[dict] = []
fitted_models: dict[str, object] = {}

for name, model in models_registry.items():
    model.fit(X_train_scaled, y_train)  # type: ignore[union-attr]
    fitted_models[name] = model

    metrics = evaluate_model(model, X_test_scaled, y_test, model_name=name)

    cv_scores = cross_val_score(
        model, X_train_scaled, y_train, cv=5, scoring="r2"
    )
    metrics["CV R² (mean)"] = round(float(cv_scores.mean()), 4)
    metrics["CV R² (std)"] = round(float(cv_scores.std()), 4)
    results_list.append(metrics)

# Comparison table
comparison_df = (
    pd.DataFrame(results_list)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)
comparison_df = comparison_df.rename(columns={"R2": "Test R²"})
print("\nModel Comparison Table (sorted by Test R²):")
display(comparison_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Model Performance Comparison", fontsize=14, fontweight="bold", color="#e2e8f0")

bar_colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(comparison_df)))

# R² bar chart
ax1 = axes[0]
bars = ax1.barh(
    comparison_df["Model"].tolist()[::-1],
    comparison_df["Test R²"].tolist()[::-1],
    color=bar_colors,
    edgecolor="none",
)
ax1.set_title("Test R² (higher = better)", color="#e2e8f0")
ax1.set_xlabel("R²", color="#a0aec0")
ax1.set_xlim(0, 1.05)
for bar, val in zip(bars, comparison_df["Test R²"].tolist()[::-1]):
    ax1.text(
        float(val) + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{float(val):.4f}",
        va="center",
        fontsize=8,
        color="#e2e8f0",
    )

# RMSE bar chart
ax2 = axes[1]
bars2 = ax2.barh(
    comparison_df["Model"].tolist()[::-1],
    comparison_df["RMSE"].tolist()[::-1],
    color=bar_colors,
    edgecolor="none",
)
ax2.set_title("RMSE (lower = better)", color="#e2e8f0")
ax2.set_xlabel("RMSE", color="#a0aec0")
for bar, val in zip(bars2, comparison_df["RMSE"].tolist()[::-1]):
    ax2.text(
        float(val) + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{float(val):.4f}",
        va="center",
        fontsize=8,
        color="#e2e8f0",
    )

plt.tight_layout()
plt.savefig(IMAGES_DIR / "model_comparison.png", bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

---
## Section 6 — Hyperparameter Tuning (GridSearchCV)

The best-performing tree-based model from Section 5 is selected for `GridSearchCV` tuning.  
On this dataset, **Random Forest** or **Gradient Boosting** typically tops the leaderboard.

### Grid design rationale
- **`n_estimators`:** More trees improve stability at the cost of compute time. `[100, 200, 300]` is sufficient for 200 rows.
- **`max_depth`:** Controls tree complexity. `None` (unlimited) can overfit; `5` and `10` add regularisation.
- **`min_samples_split`:** Minimum samples required to split a node. Higher values prevent over-splitting on sparse regions.
- **`cv=5`:** 5-fold CV is standard. With 160 training rows, this gives ~128 train / 32 val per fold — reasonable.
- **`scoring="r2"`:** Consistent with the primary evaluation metric in Section 5.

In [ ]:
# Select best tree-based model
tree_preference = ["Random Forest", "Gradient Boosting", "XGBoost", "Decision Tree"]
best_tree_name = None
for candidate in comparison_df["Model"].tolist():
    if candidate in tree_preference:
        best_tree_name = candidate
        break

if best_tree_name is None:
    best_tree_name = comparison_df["Model"].iloc[0]

print(f"Selected for tuning: {best_tree_name}")

# Define grid based on selected model
if best_tree_name == "Random Forest":
    param_grid = {
        "n_estimators": [100, 200, 300],
        "max_depth": [None, 5, 10],
        "min_samples_split": [2, 5, 10],
    }
    base_model = RandomForestRegressor(random_state=RANDOM_STATE)

elif best_tree_name == "Gradient Boosting":
    param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [3, 5],
        "learning_rate": [0.05, 0.1, 0.2],
    }
    base_model = GradientBoostingRegressor(random_state=RANDOM_STATE)

elif best_tree_name == "XGBoost":
    param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [3, 5],
        "learning_rate": [0.05, 0.1],
    }
    base_model = XGBRegressor(random_state=RANDOM_STATE, verbosity=0)

else:  # Decision Tree fallback
    param_grid = {
        "max_depth": [3, 5, 10, None],
        "min_samples_split": [2, 5, 10],
    }
    base_model = DecisionTreeRegressor(random_state=RANDOM_STATE)

print(f"\nGrid: {param_grid}")
total_fits = 1
for v in param_grid.values():
    total_fits *= len(v)
print(f"Total combinations: {total_fits}  |  CV folds: 5  |  Total fits: {total_fits * 5}")

In [ ]:
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1,
    refit=True,
)
grid_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters : {grid_search.best_params_}")
print(f"Best CV R²      : {float(grid_search.best_score_):.4f}")

best_model = grid_search.best_estimator_
tuned_metrics = evaluate_model(
    best_model, X_test_scaled, y_test,
    model_name=f"{best_tree_name} (Tuned)"
)
print(f"\nTuned Test R²  : {tuned_metrics['R2']:.4f}")
print(f"Tuned Test RMSE: {tuned_metrics['RMSE']:.4f}")
print(f"Tuned Test MAE : {tuned_metrics['MAE']:.4f}")

---
## Section 7 — Model Evaluation & Visualisations

Four diagnostic plots:
1. **Actual vs Predicted** — points should cluster tightly around the 45° identity line
2. **Residual plot** — residuals should scatter randomly around zero (no pattern = no systematic bias)
3. **Residual distribution** — should be roughly bell-shaped and centred on zero
4. **Feature importances** — for the tuned tree model (unitless, relative)

> **numpy float pitfall (Tasks 2/3 carry-over):** All numpy scalars passed to `plt.text()`, `ax.annotate()`, or formatted in f-strings are wrapped with `float()` or `int()` to avoid `numpy.float64` matplotlib annotation issues.

In [ ]:
y_pred = best_model.predict(X_test_scaled)
residuals = y_test.values - y_pred

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"{best_tree_name} (Tuned) — Evaluation Plots", fontsize=14, fontweight="bold", color="#e2e8f0")

# 1. Actual vs Predicted
ax1 = axes[0, 0]
ax1.scatter(y_test.values, y_pred, alpha=0.6, color="#a78bfa", s=40, edgecolors="none")
min_val = float(min(y_test.min(), y_pred.min()))
max_val = float(max(y_test.max(), y_pred.max()))
ax1.plot([min_val, max_val], [min_val, max_val], "--", color="#f59e0b", linewidth=2, label="45° line")
ax1.set_xlabel("Actual Sales", color="#a0aec0")
ax1.set_ylabel("Predicted Sales", color="#a0aec0")
ax1.set_title("Actual vs Predicted", color="#e2e8f0")
ax1.legend()
ax1.text(
    float(min_val) + 0.5, float(max_val) - 1.5,
    f"R² = {float(tuned_metrics['R2']):.4f}",
    fontsize=10, color="#34d399"
)

# 2. Residual plot
ax2 = axes[0, 1]
ax2.scatter(y_pred, residuals, alpha=0.6, color="#60a5fa", s=40, edgecolors="none")
ax2.axhline(0, color="#f59e0b", linestyle="--", linewidth=2)
ax2.set_xlabel("Predicted Sales", color="#a0aec0")
ax2.set_ylabel("Residuals", color="#a0aec0")
ax2.set_title("Residual Plot", color="#e2e8f0")

# 3. Residual distribution
ax3 = axes[1, 0]
sns.histplot(residuals, kde=True, ax=ax3, color="#34d399", bins=20, alpha=0.7)
ax3.axvline(0, color="#f59e0b", linestyle="--", linewidth=2)
ax3.set_xlabel("Residuals", color="#a0aec0")
ax3.set_title("Residual Distribution", color="#e2e8f0")
ax3.text(
    float(residuals.min()) * 0.8,
    ax3.get_ylim()[1] * 0.85,
    f"mean={float(residuals.mean()):.3f}\nstd={float(residuals.std()):.3f}",
    fontsize=9, color="#e2e8f0"
)

# 4. Feature importances (tree model)
ax4 = axes[1, 1]
if hasattr(best_model, "feature_importances_"):
    imp = best_model.feature_importances_
    imp_colors = ["#a78bfa", "#60a5fa", "#34d399"]
    bars4 = ax4.bar(FEATURE_COLS, imp, color=imp_colors, edgecolor="none")
    ax4.set_title(f"{best_tree_name} Feature Importances", color="#e2e8f0")
    ax4.set_ylabel("Importance (unitless)", color="#a0aec0")
    for bar, v in zip(bars4, imp):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            float(v) + 0.005,
            f"{float(v):.3f}",
            ha="center", va="bottom", fontsize=10, color="#e2e8f0"
        )
else:
    # Linear model — show coefficients instead
    coefs = np.array(best_model.coef_)
    coef_colors = ["#a78bfa" if c > 0 else "#ef4444" for c in coefs]
    bars4 = ax4.bar(FEATURE_COLS, coefs, color=coef_colors, edgecolor="none")
    ax4.set_title("Linear Model Coefficients (scaled units)", color="#e2e8f0")
    ax4.set_ylabel("Coefficient", color="#a0aec0")
    ax4.axhline(0, color="white", linewidth=0.8)
    # Note: linear coefficients are on scaled feature units here

plt.tight_layout()
plt.savefig(IMAGES_DIR / "evaluation_plots.png", bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

---
## Section 8 — Single-Point Prediction

### Why build input as a named-column DataFrame?

If we passed `[tv, radio, newspaper]` as a raw list or numpy array to `scaler.transform()`, it would work today — but silently break the moment column order changes between model versions.  Named-column DataFrames make such bugs **loud** (a `KeyError`) instead of silently corrupting the prediction.  This is a production-ready pattern.

**Demo prediction:** TV=230, Radio=40, Newspaper=15 — a relatively large TV + moderate Radio campaign.

In [ ]:
demo_prediction = predict_sales(
    tv=230.0,
    radio=40.0,
    newspaper=15.0,
    model=best_model,
    scaler=scaler,
    feature_names=FEATURE_COLS,
)

print("Single-Point Prediction Demo")
print("=============================")
print(f"  TV        : $230K")
print(f"  Radio     : $40K")
print(f"  Newspaper : $15K")
print(f"  Predicted Sales: {demo_prediction:.2f} units")

---
## Section 9 — Batch Forecasting

The `forecast_sales()` function accepts a DataFrame of upcoming campaign budgets and returns predictions as a new column `Predicted_Sales`.  This is the pattern used in the Streamlit dashboard to populate the batch forecast table.

In [ ]:
upcoming_campaigns = pd.DataFrame({
    "TV": [230.0, 150.0, 80.0],
    "Radio": [40.0, 25.0, 10.0],
    "Newspaper": [15.0, 30.0, 50.0],
})

forecast_result = forecast_sales(
    df_new=upcoming_campaigns,
    model=best_model,
    scaler=scaler,
    feature_names=FEATURE_COLS,
)
forecast_result["Predicted_Sales"] = forecast_result["Predicted_Sales"].round(2)

print("Batch Forecast Results:")
display(forecast_result)

---
## Section 10 — Business Insights

All insights are **generated programmatically** from model outputs — no hardcoded prose.

### Two ranking methods
1. **Pearson correlation** — captures linear relationship between each channel and Sales; ignores interaction effects between channels.
2. **Tree feature importances** — capture non-linear and interaction effects; unitless (can't be compared across models as absolute magnitudes).

### ROI proxy
We use **Linear Regression coefficients on original scale** as the ROI proxy.  
Interpretation: a coefficient of X means "each additional $1,000 of spend in this channel is associated with X additional units of Sales, holding other channels constant".  
Tree importance is unitless and doesn't give this marginal interpretation.

### Disagreement between correlation and importance
It's common for Newspaper to rank last in both — but if tree importance shows Radio higher than TV (while correlation shows TV higher), that can happen because Radio spend often operates in a low-absolute-spend range where each marginal dollar has a larger relative effect, while TV is already saturating.

In [ ]:
# 1. Pearson correlation ranking
corr_series = df[FEATURE_COLS].corrwith(df[TARGET_COL]).sort_values(ascending=False)
print("Channel correlation with Sales (Pearson r):")
for ch, val in corr_series.items():
    print(f"  {ch:<12}: {float(val):.4f}")

# 2. Linear regression coefficients (unscaled back to original units = ROI proxy)
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Unscale: coefficient on original scale = coef_scaled / std_of_feature
raw_coefs = lr_model.coef_ / scaler.scale_
coef_df = pd.DataFrame(
    {"Channel": FEATURE_COLS, "ROI_proxy": raw_coefs}
).sort_values("ROI_proxy", ascending=False)

print("\nLinear model ROI proxy (Sales units per $000 spend, other channels held constant):")
for _, row in coef_df.iterrows():
    print(f"  {row['Channel']:<12}: {float(row['ROI_proxy']):.4f}")

# 3. Tree feature importances
if hasattr(best_model, "feature_importances_"):
    imp_df = pd.DataFrame(
        {"Channel": FEATURE_COLS, "Importance": best_model.feature_importances_}
    ).sort_values("Importance", ascending=False)
    print(f"\n{best_tree_name} feature importances (unitless):")
    for _, row in imp_df.iterrows():
        print(f"  {row['Channel']:<12}: {float(row['Importance']):.4f}")

# 4. Weakest channel and reallocation suggestion
weakest_ch = coef_df.iloc[-1]["Channel"]
strongest_ch = coef_df.iloc[0]["Channel"]
print(f"\n>>> Directional insight: '{weakest_ch}' has the lowest ROI proxy.")
print(f"    The model suggests reallocating {weakest_ch} budget toward {strongest_ch}.")
print(f"    ⚠️  CAVEAT: 200 rows, 3 features — treat as directional only, not prescriptive.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Business Insights — Channel Analysis", fontsize=14, fontweight="bold", color="#e2e8f0")

insight_colors = ["#a78bfa", "#60a5fa", "#34d399"]

# Correlation bar
ax1 = axes[0]
sorted_channels = corr_series.index.tolist()
sorted_vals = corr_series.values.tolist()
bars_c = ax1.bar(sorted_channels, sorted_vals, color=insight_colors, edgecolor="none")
ax1.set_title("Pearson Correlation with Sales", color="#e2e8f0")
ax1.set_ylabel("Correlation", color="#a0aec0")
ax1.set_ylim(0, 1)
for bar, val in zip(bars_c, sorted_vals):
    ax1.text(bar.get_x() + bar.get_width() / 2, float(val) + 0.01, f"{float(val):.3f}",
             ha="center", va="bottom", fontsize=10, color="#e2e8f0")

# ROI proxy bar
ax2 = axes[1]
roi_channels = coef_df["Channel"].tolist()
roi_vals = coef_df["ROI_proxy"].tolist()
bars_r = ax2.bar(roi_channels, roi_vals, color=insight_colors, edgecolor="none")
ax2.set_title("ROI Proxy: Sales per $000 Spend", color="#e2e8f0")
ax2.set_ylabel("Δ Sales per $000", color="#a0aec0")
for bar, val in zip(bars_r, roi_vals):
    ax2.text(bar.get_x() + bar.get_width() / 2, float(val) + 0.001, f"{float(val):.4f}",
             ha="center", va="bottom", fontsize=10, color="#e2e8f0")

plt.tight_layout()
plt.savefig(IMAGES_DIR / "business_insights.png", bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

---
## Section 11 — Model Persistence & Reload Verification

Three artifacts are saved with `joblib.dump`:  
- `model.pkl` — best tuned model  
- `scaler.pkl` — fitted StandardScaler  
- `features.pkl` — ordered list of feature names  

After saving, we **reload all three** and assert that the reloaded model's prediction on a test row matches the in-memory model's prediction.  This proves that `app.py` will get the same results as this notebook.

In [ ]:
feature_names = FEATURE_COLS

save_artifacts(best_model, scaler, feature_names)
print("Artifacts saved:")
for fname in ["model.pkl", "scaler.pkl", "features.pkl"]:
    size_kb = Path(fname).stat().st_size / 1024
    print(f"  {fname:<15}: {size_kb:.1f} KB")

In [ ]:
# Reload and verify
reloaded_model, reloaded_scaler, reloaded_features = load_artifacts()

print(f"Reloaded model type  : {type(reloaded_model).__name__}")
print(f"Reloaded features    : {reloaded_features}")

# Prediction on first test row — must match exactly
test_row = X_test.iloc[[0]].copy()

pred_original = float(best_model.predict(
    pd.DataFrame(scaler.transform(test_row), columns=FEATURE_COLS)
)[0])

pred_reloaded = predict_sales(
    tv=float(test_row["TV"].iloc[0]),
    radio=float(test_row["Radio"].iloc[0]),
    newspaper=float(test_row["Newspaper"].iloc[0]),
    model=reloaded_model,
    scaler=reloaded_scaler,
    feature_names=reloaded_features,
)

assert abs(pred_original - pred_reloaded) < 1e-6, (
    f"MISMATCH: original={pred_original:.6f}, reloaded={pred_reloaded:.6f}"
)

print(f"\nIn-memory prediction : {pred_original:.6f}")
print(f"Reloaded prediction  : {pred_reloaded:.6f}")
print("✅ Reload verification PASSED — app.py will produce identical predictions")

---
## Summary

| Item | Result |
|------|--------|
| Dataset | 200 rows × 4 columns, 0 nulls, 0 duplicates |
| Best model | Selected by GridSearchCV from 7 algorithms |
| Best CV R² | Reported by GridSearchCV above |
| Strongest channel | TV (highest correlation + ROI proxy) |
| Weakest channel | Newspaper (lowest correlation + ROI proxy) |
| Artifacts | model.pkl, scaler.pkl, features.pkl — verified |
| Dashboard | `streamlit run app.py` |

**Next step:** Run `streamlit run app.py` to launch the interactive dashboard.

---
*CodeAlpha Data Science Internship — Task 4 | Sales Prediction System*